# Comparison changing relative importance between force weights and energy weights

In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write
from google.colab import drive
from mace.cli.run_train import main

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

# Define file paths
outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/force1000"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/force1000"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/force1000"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::1", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_force1000",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",
    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=foundation", # try "isolated" after adding isolated atom energies to training set?
               #"mp" means from materials project (used to train foundation model)
    #"--model=MACE",
    #"--num_interactions=2", # keep default


    "--stress_key=stress",
    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=50",
    "--patience=10",
    "--batch_size=5",       # was 10
    "--valid_batch_size=5", # was 10
    "--device=cuda",
    "--default_dtype=float32", # less memory than 64

    "--r_max=5.0", #could be increased??


    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",

    # use a pre-trained ("foundation") model:
    "--foundation_model=small",           # Using different base model since the small model is trainmed with L_max=0 and we are trying to add further terms of the spherical harmonics
    "--multiheads_finetuning=True",        # Enable recommended Multihead Replay (for generalizable models; but expensive)
    "--pt_train_file=mp",                 # Required for Multihead Replay


    f"--atomic_numbers={my_atomic_numbers}", #[1, 6, 7, 50, 53]
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0", ##############
    "--forces_weight=1000.0", #############

#OPTIMIZER
    "--optimizer=adamw",  #AdamW - optimizer
    "--amsgrad", #AMSGrad variant --> ensure stability
    "--weight_decay=5e-7",


#EMA/SWA
#optimization technique and loss function
#EMA: exponential smoothing
#SWA: stochastic weight averaging

#ema: Enables Exponential Moving Average during training.
#Instead of saving only the model weights from the current optimization step, it maintains a running average of the model parameters.
#This produces a smoother, more stable set of parameters, reducing variance and helping the model generalize better on unseen structures.
    "--ema",
    "--ema_decay=0.99",  #decay rate

#swa: Enables Stochastic Weight Averaging, a technique that averages model weights sampled across multiple late stage epochs
#find a broader, flatter local minimum in the loss landscape.
    "--swa",
    "--start_swa=30", #averaging will begin at epoch start_swa

#importance
    "--swa_forces_weight=1000.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",


#Dynamically normalizes the loss contributions based on the Root Mean Square (RMS) of the atomic forces in the dataset.
#This prevents large force magnitudes from dominating the gradients unevenly across different atomic configurations.
    "--scaling=rms_forces_scaling",


]

sys.argv = custom_args


sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

/usr/local/lib/python3.13/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading and subsampling OUTCAR...
Extracted 3160 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 2528 train, 316 valid, and 316 test frames.
2026-09-21 15:13:50.766 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


...

2026-09-21 17:38:36.099 INFO: Skipping evaluation of valid_pt_head (in skip_heads list)


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+---------------------------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % | RMSE Stress (Virials) / meV / A (A^3) |
+---------------+---------------------+------------------+-------------------+---------------------------------------+
| train_Default |            0.5      |         22.0     |          6.53     |                     4.6               |
| valid_Default |            0.7      |         41.5     |         12.00     |                     4.4               |
+---------------+---------------------+------------------+-------------------+---------------------------------------+


2026-09-21 17:38:36.102 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+---------------------------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % | RMSE Stress (Virials) / meV / A (A^3) |
+---------------+---------------------+------------------+-------------------+---------------------------------------+
| train_Default |            0.5      |         22.0     |          6.53     |                     4.6               |
| valid_Default |            0.7      |         41.5     |         12.00     |                     4.4               |
+---------------+---------------------+------------------+-------------------+---------------------------------------+


DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on train_pt_head dataset
DEBUG:root:Running inference on valid_pt_head dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-21 17:40:33.847 INFO: Done
Training completed!


Simulating trajectory

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

#  Mount Google Drive
drive.mount('/content/drive')

#  Define File Paths
model_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/force1000/model_3000_pts_force1000_run-1_stagetwo.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/force/md_trajectory_force1000_2000.extxyz"

output_dir = os.path.dirname(output_traj_path)
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured output directory exists: {output_dir}")

#  Load the initial structure (frame 0)
print("Loading initial structure...")
atoms = read(init_structure_path, index=0)

#  Initialize the MACE Calculator with compiled model
print(f"Loading compiled MACE model from: {model_path}...")
calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

#  Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

# Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

# 7. Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

/usr/local/lib/python3.13/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Mounted at /content/drive
Ensured output directory exists: /content/drive/MyDrive/MDProject/results/force
Loading initial structure...
Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/force1000/model_3000_pts_force1000_run-1_stagetwo.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...


/tmp/ipykernel_665/1579196522.py:43: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    0 | Temp:  336.9 K | Epot: -479.144 eV | Total E: -474.790 eV
Step:    1 | Temp:  309.8 K | Epot: -478.765 eV | Total E: -474.760 eV
Step:    2 | Temp:  256.0 K | Epot: -478.012 eV | Total E: -474.704 eV
Step:    3 | Temp:  236.2 K | Epot: -477.757 eV | Total E: -474.704 eV
Step:    4 | Temp:  255.8 K | Epot: -478.046 eV | Total E: -474.740 eV
Step:    5 | Temp:  272.2 K | Epot: -478.266 eV | Total E: -474.747 eV
Step:    6 | Temp:  260.2 K | Epot: -478.086 eV | Total E: -474.723 eV
Step:    7 | Temp:  229.5 K | Epot: -477.664 eV | Total E: -474.697 eV
Step:    8 | Temp:  218.8 K | Epot: -477.490 eV | Total E: -474.662 eV
Step:    9 | Temp:  252.3 K | Epot: -477.947 eV | Total E: -474.685 eV
Step:   10 | Temp:  310.4 K | Epot: -478.748 eV | Total E: -474.736 eV
Step:   11 | Temp:  340.3 K | Epot: -479.155 eV | Total E: -474.756 eV
Step:   12 | Temp:  318.7 K | Epot: -478.841 eV | Total E: -474.721 eV
Step:   13 | Temp:  281.2 K | Epot: -478.302 eV | Total E: -474.667 eV
Step: 

Full analysis

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/force/md_trajectory_force1000_2000.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/force/force1000"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



[Total Energy] t0 = 630 (frame 630), g = 41.52, N_eff = 33.0 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8166
Area (180° peak): 0.1834
Max v dataset: 0.14172885999999885
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1416.37 K


Force-energy importance: 10:1

In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write
from google.colab import drive
from mace.cli.run_train import main

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

# Define file paths
outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/force10"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/force10"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/force10"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::1", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")


print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_force10",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",
    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=foundation", # try "isolated" after adding isolated atom energies to training set?
               #"mp" means from materials project (used to train foundation model)
    #"--model=MACE",
    #"--num_interactions=2", # keep default


    "--stress_key=stress",
    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=50",
    "--patience=10",
    "--batch_size=5",       # was 10
    "--valid_batch_size=5", # was 10
    "--device=cuda",
    "--default_dtype=float32", # less memory than 64

    "--r_max=5.0", #could be increased??


    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",

    # use a pre-trained ("foundation") model:
    "--foundation_model=small",           # Using different base model since the small model is trainmed with L_max=0 and we are trying to add further terms of the spherical harmonics
    "--multiheads_finetuning=True",        # Enable recommended Multihead Replay (for generalizable models; but expensive)
    "--pt_train_file=mp",                 # Required for Multihead Replay


    f"--atomic_numbers={my_atomic_numbers}", #[1, 6, 7, 50, 53]
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0", ##############
    "--forces_weight=10.0", #############

#OPTIMIZER
    "--optimizer=adamw",  #AdamW - optimizer
    "--amsgrad", #AMSGrad variant --> ensure stability
    "--weight_decay=5e-7",


#EMA/SWA
#optimization technique and loss function
#EMA: exponential smoothing
#SWA: stochastic weight averaging

#ema: Enables Exponential Moving Average during training.
#Instead of saving only the model weights from the current optimization step, it maintains a running average of the model parameters.
#This produces a smoother, more stable set of parameters, reducing variance and helping the model generalize better on unseen structures.
    "--ema",
    "--ema_decay=0.99",  #decay rate

#swa: Enables Stochastic Weight Averaging, a technique that averages model weights sampled across multiple late stage epochs
#find a broader, flatter local minimum in the loss landscape.
    "--swa",
    "--start_swa=30", #averaging will begin at epoch start_swa

#importance
    "--swa_forces_weight=10.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",


#Dynamically normalizes the loss contributions based on the Root Mean Square (RMS) of the atomic forces in the dataset.
#This prevents large force magnitudes from dominating the gradients unevenly across different atomic configurations.
    "--scaling=rms_forces_scaling",
    "--restart_latest"


]

sys.argv = custom_args


sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

/usr/local/lib/python3.13/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Mounted at /content/drive
Reading and subsampling OUTCAR...
Extracted 3160 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 2528 train, 316 valid, and 316 test frames.
2026-09-25 16:53:29.262 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


...

2026-09-25 18:01:54.666 INFO: Skipping evaluation of valid_pt_head (in skip_heads list)


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+---------------------------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % | RMSE Stress (Virials) / meV / A (A^3) |
+---------------+---------------------+------------------+-------------------+---------------------------------------+
| train_Default |            0.4      |         21.4     |          6.35     |                     1.2               |
| valid_Default |            0.6      |         41.9     |         12.10     |                     1.0               |
+---------------+---------------------+------------------+-------------------+---------------------------------------+


2026-09-25 18:01:54.672 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+---------------------------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % | RMSE Stress (Virials) / meV / A (A^3) |
+---------------+---------------------+------------------+-------------------+---------------------------------------+
| train_Default |            0.4      |         21.4     |          6.35     |                     1.2               |
| valid_Default |            0.6      |         41.9     |         12.10     |                     1.0               |
+---------------+---------------------+------------------+-------------------+---------------------------------------+


DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on train_pt_head dataset
DEBUG:root:Running inference on valid_pt_head dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-25 18:03:56.677 INFO: Done
Training completed!


Simulating trajectory

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

#  Mount Google Drive
drive.mount('/content/drive')

#  Define File Paths
model_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/force10/model_3000_pts_force10_run-1.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/force/md_trajectory_force10_2000.extxyz"

# Automatically create the output directory path if it doesn't exist ---
output_dir = os.path.dirname(output_traj_path)
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured output directory exists: {output_dir}")


#  Load the initial structure (frame 0)
print("Loading initial structure...")
atoms = read(init_structure_path, index=0)

#  Initialize the MACE Calculator with compiled model
print(f"Loading compiled MACE model from: {model_path}...")
calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

#  Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

#  Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

#  Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured output directory exists: /content/drive/MyDrive/MDProject/results/force
Loading initial structure...
Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/force10/model_3000_pts_force10_run-1.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...
Step:    0 | Temp:  297.3 K | Epot: -479.116 eV | Total E: -475.273 eV


/tmp/ipykernel_665/1985797269.py:43: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    1 | Temp:  276.6 K | Epot: -478.841 eV | Total E: -475.266 eV
Step:    2 | Temp:  234.7 K | Epot: -478.268 eV | Total E: -475.234 eV
Step:    3 | Temp:  204.9 K | Epot: -477.877 eV | Total E: -475.228 eV
Step:    4 | Temp:  202.1 K | Epot: -477.864 eV | Total E: -475.251 eV
Step:    5 | Temp:  211.7 K | Epot: -478.017 eV | Total E: -475.280 eV
Step:    6 | Temp:  214.0 K | Epot: -478.053 eV | Total E: -475.287 eV
Step:    7 | Temp:  203.9 K | Epot: -477.908 eV | Total E: -475.272 eV
Step:    8 | Temp:  197.1 K | Epot: -477.801 eV | Total E: -475.253 eV
Step:    9 | Temp:  215.0 K | Epot: -478.034 eV | Total E: -475.254 eV
Step:   10 | Temp:  253.0 K | Epot: -478.556 eV | Total E: -475.287 eV
Step:   11 | Temp:  282.9 K | Epot: -478.944 eV | Total E: -475.288 eV
Step:   12 | Temp:  281.4 K | Epot: -478.911 eV | Total E: -475.273 eV
Step:   13 | Temp:  260.5 K | Epot: -478.618 eV | Total E: -475.251 eV
Step:   14 | Temp:  247.8 K | Epot: -478.434 eV | Total E: -475.231 eV
Step: 

Full analysis

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/force/md_trajectory_force10_2000.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/force/force10"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

[Total Energy] t0 = 5 (frame 5), g = 38.20, N_eff = 52.3 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8086
Area (180° peak): 0.1914
Max v dataset: 0.12887840000000006
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1295.43 K


In [ ]:
! pip install ase
! pip install pymbar
! pip install mace-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.0 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39523 sha256=9201e31de8b78734214a73dff308057e10101582be160424af6762605a755893
  Stored in directory: /root/.cache/pip/wheels/05/2e/7e/39f980a5a0755f

In [ ]:
import os
import automatic_analysis5
from google.colab import drive

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

input_traj_dir = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr3_10k.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/correlation/corr3_10k"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=1000, timestep_in_fs=0.5, window_size=20)

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



Mounted at /content/drive
[Total Energy] t0 = 9980 (frame 9980), g = 1.11, N_eff = 19.7 / 10001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.9834
Area (180° peak): 0.0166
Max v dataset: 0.4221967800000001
Processed 10001 frames successfully.
Processed 10001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1482.16 K


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

In [ ]:
! pip install ase
! pip install pymbar
! pip install mace-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.5/457.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.1 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39523 sha256=ec23290ca3dd545caa73577c5cb08909e048aa0a2baef28ed1b0b9ca2c306c0e
  Stored in directory: /root/.cache/pip/wheels/05/2e/7e/39f980a5a0755f

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load the full 50-epoch MACE training log
log_file_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/force10/model_3000_pts_force10_run-1_train.txt"
data = pd.read_csv(log_file_path, delim_whitespace=True, comment='#')

data = data[data['epoch'] <= 30]

# 2. Extract the critical columns (now limited to 30 epochs)
epochs = data['epoch']
train_force = data['rmse_f_train'] * 1000
valid_force = data['rmse_f_val'] * 1000
train_energy = data['rmse_e_train'] * 1000
valid_energy = data['rmse_e_val'] * 1000

# 3. Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Force Plot ---
ax1.plot(epochs, train_force, label='Train Forces', color='blue', linewidth=2)
ax1.plot(epochs, valid_force, label='Validation Forces', color='red', linestyle='--', linewidth=2)
ax1.set_title('Force RMSE (Epochs 1-30)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Force RMSE (meV/Å)')
ax1.set_yscale('log')
ax1.legend()
ax1.grid(True, which="both", ls="-", alpha=0.2)

# --- Energy Plot ---
ax2.plot(epochs, train_energy, label='Train Energy', color='blue', linewidth=2)
ax2.plot(epochs, valid_energy, label='Validation Energy', color='red', linestyle='--', linewidth=2)
ax2.set_title('Energy RMSE (Epochs 1-30)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Energy RMSE (meV/atom)')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, which="both", ls="-", alpha=0.2)

plt.tight_layout()
plt.savefig('/content/mace_training_diagnostics_30.png', dpi=300)
plt.show()

/tmp/ipykernel_4142/2350799795.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(log_file_path, delim_whitespace=True, comment='#')


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/force10/model_3000_pts_force10_run-1_train.txt'